In [ ]:
from adapt_eeg.auto_annotation.classes import SyllableIdentifyParams
from adapt_eeg.auto_annotation.syllable_from_textgrid import load_textgrid_syllable

from adapt_eeg.constants import POEMS_CONFIG

poem1_config = POEMS_CONFIG[0]
syllable_params = SyllableIdentifyParams()

syllablized_audio = load_textgrid_syllable(
    poem1_config["audio_url"],
    poem1_config["textgrid_url"],
    intensity_floor_hz=syllable_params.intensity_floor_hz,
    pitch_timestep_s=syllable_params.pitch_time_step_s,
    pitch_range_hz=syllable_params.pitch_range_hz,
    silence_threshold_relative_db=syllable_params.silence_threshold_relative_db,
    syllable_pitch_percentile=syllable_params.pitch_percentile,
    min_nucleus_prominence_db=syllable_params.min_nucleus_prominence_db,
)

syllablized_audio = syllablized_audio[poem1_config["intro_end"] :]  # cut off the intro

In [ ]:
from adapt_eeg.auto_annotation.classes import TranscriptedSyllable


prev = None
for syl in syllablized_audio.syllables:
    assert isinstance(syl, TranscriptedSyllable), "Why is textgrid loading doesnt have transcription"
    if syl.transcription == "mackerel":
        break
    prev = syl

In [ ]:
prev._audio

In [ ]:
from adapt_eeg.auto_annotation.classes import StressIdentifyParams
from adapt_eeg.auto_annotation.stress_identifier import flag_stressed_syllables

params = StressIdentifyParams(
    pitch_range_hz=(75.0, 500.0),
    prominence_neighbor_half_window_s=0.05,
    prominence_min_nucleus_distance_s=0.05,
    min_pitch_prominence_hz=0.52,  # 2
    min_intensity_prominence_db=0.28,  # 5
    duration_neighbor_radius=3,
    stress_weights=(5, 3, 2),
    syllables_per_line=10,
    stressed_syllables_per_line=5,
)

stress_flags = flag_stressed_syllables(syllablized_audio, params)
print(sum(stress_flags))

In [ ]:
from dataclasses import dataclass

import numpy as np

from adapt_eeg.auto_annotation.classes import Syllable


@dataclass(frozen=True)
class ScannedVerseLine:
    syllables: list[Syllable]
    stress_flags: list[bool]

    def __post_init__(self):
        assert len(self.syllables) == len(self.stress_flags)

    def stress_std(self):
        stressed_syllables = [
            syllable
            for syllable, stressed in zip(self.syllables, self.stress_flags)
            if stressed
        ]
        
        stressed_nucleus = [s.nucleus for s in stressed_syllables]
        
        dists = np.diff(stressed_nucleus)
        std_dist = np.std(dists)
        
        return std_dist

In [ ]:
lines = []

for i in range(0, len(stress_flags), 10):
    line_stress_flags = stress_flags[i : i + 10]
    line_syllables = syllablized_audio.syllables[i : i + 10]

    line = ScannedVerseLine(line_syllables, line_stress_flags)
    lines.append(line)
    


In [ ]:
from adapt_eeg.auto_annotation.classes import TranscriptedSyllable


for line in lines:
    for syl in line.syllables:
        assert isinstance(syl, TranscriptedSyllable)
        print(syl.transcription, end=" ")
    print()

In [ ]:
from matplotlib import pyplot as plt

stds = [line.stress_std() for line in lines]

plt.hist(stds, bins=30)
plt.show()